# 05 — Optimizer Gantt + cost analysis

Solves the 96-slot MILP for the first test day, compares against the naïve baseline, and plots a Gantt + price overlay.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from aerogrid.optimizer import solve_proactive_schedule
from aerogrid.price_oracle import load_price_history, make_oracle
from aerogrid.behavioral_predictor import load_onsets, make_predictor
from aerogrid.config import NYISO_TEST_START, APPLIANCES, SLOT_MINUTES

prices_df = load_price_history()
now = NYISO_TEST_START
fc = make_oracle('naive').get_15min_forecast(now, prices_df)
prices = np.asarray(fc.median)
op = make_predictor().fit(load_onsets()).predict_all(now)
sched = solve_proactive_schedule(now, prices, op)
print(f"expected ${sched.expected_cost:.3f}  baseline ${sched.baseline_cost:.3f}  savings {sched.savings()*100:.1f}%")

In [ ]:
# Gantt chart of the schedule overlaid with the price forecast
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True,
                               gridspec_kw={'height_ratios':[3,2]})
hrs = np.arange(96)/4
colors = {'dishwasher':'#1f77b4','washing_machine':'#2ca02c'}
# EV stacked bars
ax1.bar(hrs, sched.ev_power_kw, width=0.25, color='#ff7f0e', alpha=0.7, label='EV (kW)')
for t in sched.tasks:
    spec = APPLIANCES[t.appliance]
    ax1.barh(y=spec.rated_kw/2, width=t.slots*0.25, left=t.start_slot*0.25,
             height=0.8, color=colors[t.appliance], alpha=0.6, label=t.appliance)
ax1.set_ylabel("load (kW)"); ax1.legend(loc='upper right'); ax1.set_title('schedule — first test day')
ax2.plot(hrs, prices, color='k', label='LBMP forecast ($/MWh)')
ax2.set_ylabel("$/MWh"); ax2.set_xlabel("hour"); ax2.legend()
plt.tight_layout(); plt.show()